# D0 — document, scope, context and extraction contracts

This notebook accompanies the neutral contract layer in #11, not a document ingestion pipeline.
It uses only core values: no Git/database handle, LangChain, model credentials or network.

We will distinguish a logical document, a captured revision, reusable bytes, collection
membership and a chunk occurrence. Then we will show why old source validity does not
establish old knowledge. Finally we will narrow trusted scope, account for actual context
exposure and distinguish candidate/assertion intent from incidental processing runs.
See the document, graph/context and extraction contract specifications under `docs/`.

In [ ]:
from datetime import UTC, datetime

from pydantic import ValidationError

from synirmos.core.contracts import ArtifactSpec, SourceKey, json_fingerprint
from synirmos.core.documents import (
    ChunkContent,
    ChunkOccurrence,
    ContentReference,
    DocumentBatch,
    DocumentCollection,
    DocumentLimits,
    DocumentRevision,
    SourceSpan,
    TextDocument,
)
from synirmos.core.temporal import ReadPoint, RevisionTiming, ValidInterval

VALID_FROM = datetime(2026, 11, 1, tzinfo=UTC)
CAPTURED_AT = datetime(2026, 11, 28, tzinfo=UTC)
prefix = b"# Notes\r\n\r\n"
line = "Ada wrote café.\r\n".encode()
raw = prefix + line + line
content = ContentReference.from_bytes(raw)
timing = RevisionTiming(
    valid=ValidInterval(start=VALID_FROM),
    observed_at=CAPTURED_AT,
    info_horizon=CAPTURED_AT,
)
documents = [
    TextDocument(key=SourceKey(source_instance=source, native_key="notes"), title="Notes")
    for source in ("memory:alice", "memory:bob")
]
revisions = [DocumentRevision(document=doc, content=content, timing=timing) for doc in documents]
assert documents[0].document_id != documents[1].document_id
assert revisions[0].revision_id != revisions[1].revision_id
assert revisions[0].content == revisions[1].content
print("Same bytes; distinct source documents and captured revisions.")

## Membership does not own identity; replay does not create a new observation

One document can belong to two collections. Neither membership nor an incidental
processing run changes its identity. A retry restores the original capture timestamps;
calling a new clock would instead describe another observation. Title changes do not
change logical identity, but captured metadata does participate in revision identity.

In [ ]:
memberships = [
    DocumentCollection(
        key=SourceKey(source_instance="collections:demo", native_key=name),
        document_ids=(documents[0].document_id,),
    )
    for name in ("team", "research")
]
assert memberships[0].collection_id != memberships[1].collection_id
assert memberships[0].document_ids == memberships[1].document_ids
retry = DocumentRevision.model_validate_json(revisions[0].model_dump_json())
assert retry == revisions[0]
assert retry.revision_id == revisions[0].revision_id
renamed = TextDocument(key=documents[0].key, title="Renamed notes")
assert renamed.document_id == documents[0].document_id
print("Two memberships; one logical document; retry retains captured identity.")

## Source validity, observation and justified knowledge are independent

Our source says the text is valid from November 1, but we captured it on November 28.
That does not prove the content was available to this application on November 1.
Backdating knowledge requires availability evidence that a source adapter must verify.
Merely supplying an evidence ID is **not authentication or permission**. None of these
times is a database commit timestamp. `ReadPoint.includes` checks one revision's time
eligibility only; it does not choose a version, resolve supersession or enforce access.

In [ ]:
yesterday_view = ReadPoint(valid_time=VALID_FROM, info_horizon=VALID_FROM)
captured_view = ReadPoint(valid_time=VALID_FROM, info_horizon=CAPTURED_AT)
assert not yesterday_view.includes(timing)
assert captured_view.includes(timing)
try:
    RevisionTiming(valid=timing.valid, observed_at=CAPTURED_AT, info_horizon=VALID_FROM)
except ValidationError:
    print("Rejected unsupported backdating, despite the old validity start.")
else:
    raise AssertionError("missing availability evidence must reject")
assert not ValidInterval(start=VALID_FROM, end=CAPTURED_AT).contains(CAPTURED_AT)

## Original-byte spans, not decoded character positions

This fixture contains UTF-8, CRLF and a repeated line. We explicitly choose the second
line using the known byte structure, not an ambiguous search for its text. A real parser
must supply source maps when headings, whitespace or escapes are transformed. Span
extraction verifies the complete original content reference before slicing.

In [ ]:
start = len(prefix) + len(line)
span = SourceSpan(
    revision_id=revisions[0].revision_id, content=content, start=start, end=start + len(line)
)
assert span.extract(raw) == line
assert len(line) > len(line.decode("utf-8"))
print(f"Second line: bytes [{span.start}, {span.end}), {span.extract(raw)!r}")

## Reuse chunk content without merging its occurrences or access

A chunker specification identifies the algorithm/library version and exact config.
Identical chunk bytes under that specification may be reused across revisions, but
each original placement has its own occurrence identity. These values do not perform
chunking or grant permission to retrieve another source's content.

In [ ]:
spec = ArtifactSpec(
    implementation="example:line-splitter",
    implementation_version="1",
    config_fingerprint=json_fingerprint({"preserve_crlf": True, "overlap": 0}),
)
chunk = ChunkContent(content=ContentReference.from_bytes(line), chunker=spec)
occurrences = [
    ChunkOccurrence(
        chunk=chunk,
        span=SourceSpan(
            revision_id=rev.revision_id, content=content, start=start, end=start + len(line)
        ),
    )
    for rev in revisions
]
assert occurrences[0].chunk.chunk_id == occurrences[1].chunk.chunk_id
assert occurrences[0].occurrence_id != occurrences[1].occurrence_id
print("Reusable chunk content; distinct source occurrences.")

## Bounded pages are not unbounded-work guarantees

Source pages validate both revision count and conservatively summed content bytes.
Adapters must still enforce the caller's actual policy; self-declared returned limits
cannot expand it. A continuation/gap is distinct from a successful empty result.
These limits do not certify parser memory, source I/O, token usage or latency.

In [ ]:
limits = DocumentLimits(max_revisions=1, max_content_bytes=len(raw))
page = DocumentBatch(revisions=(revisions[0],), limits=limits, continuation="next:opaque")
assert DocumentBatch.model_validate_json(page.model_dump_json()) == page
assert DocumentBatch(continuation="next:opaque").continuation is not None
try:
    DocumentBatch(revisions=tuple(revisions), limits=limits)
except ValidationError:
    print("Rejected a two-revision page with a one-revision limit.")
else:
    raise AssertionError("source pages must enforce declared limits")

## Trusted scope is intersected, never broadened by a tool

The following is an **explicit fixture**, not authentication or a live backend. A real
runtime authenticates policy and opens a verifiable snapshot before issuing its grant.
None means all already-granted IDs; empty means none. A request cannot change the pin.
Result caps do not establish candidate, memory or deadline guarantees.

In [ ]:
from synirmos.core.graph import (
    ApprovalStatus,
    EntityRecord,
    GraphProfile,
    Properties,
    ScopeGrant,
    ScopeRequest,
    SnapshotPin,
    Visibility,
)
from synirmos.core.limits import ReadCapabilities, ReadLimitError, ReadLimits, UnsupportedReadError


def example_artifact(name):
    return ArtifactSpec(
        implementation=f"example:{name}",
        implementation_version="1",
        config_fingerprint=json_fingerprint({"fixture": True}),
    )


graph_profile = GraphProfile(
    profile_id="profile:demo",
    version="1",
    identity_version="contracts-v1",
    mapping=example_artifact("mapper"),
    labels=("https://example/Person",),
)
pin = SnapshotPin(
    database_id="memory:fixture",
    snapshot_id="snapshot:fixture",
    provenance_fingerprint=json_fingerprint({"fixture": 1}),
    point=captured_view,
    graph_profile_fingerprint=graph_profile.fingerprint,
    ontology_release="release:fixture",
)
grant = ScopeGrant(
    principal_id="user:fixture",
    authorization_fingerprint=json_fingerprint("policy:1"),
    snapshot=pin,
    collection_ids=(memberships[0].collection_id,),
    source_instances=(documents[0].key.source_instance,),
    limits=ReadLimits(max_results=2),
)
effective = grant.restrict(
    ScopeRequest(
        point=captured_view,
        collection_ids=(memberships[0].collection_id, "not-granted"),
    )
)
assert effective.collection_ids == (memberships[0].collection_id,)
assert grant.restrict(ScopeRequest(point=captured_view, collection_ids=())).is_empty
visible = Visibility(
    source_instance=documents[0].key.source_instance,
    collection_ids=(memberships[0].collection_id,),
    approval=ApprovalStatus.APPROVED,
    timing=timing,
)
assert effective.includes(visible)
try:
    ReadCapabilities().require(ReadLimits(max_candidates=10))
except UnsupportedReadError:
    print("Candidate-work bound unsupported; output limits alone do not supply it.")
else:
    raise AssertionError("unsupported physical guarantees must reject")

## Ground canonical reuse in actual exposure; share tool budgets

The host issues a receipt only after whole-record delivery. This fixture's entity ID
is explicit, not derived from a person's name. A real reader must authenticate records,
select historical versions and filter dependencies before ranking. Changed descriptions
invalidate context identity, and repeated tool calls still spend the shared budget.

In [ ]:
from synirmos.core.context import ContextRequest, ContextResult, ExposureSet, KnowledgeQuery

known = EntityRecord(
    entity_id="entity:fixture-1",
    version_id="version:1",
    labels=("https://example/Person",),
    visibility=visible,
    properties=Properties({"name": "Ada", "description": "Fixture only"}),
)
request = ContextRequest(
    subject_id="subject:fixture",
    scope=effective,
    query=KnowledgeQuery(entity_ids=(known.entity_id,)),
)
context = ContextResult.from_records(
    request=request,
    provider=example_artifact("context"),
    entities=(known,),
)
exposed = ExposureSet(scope=effective, contexts=(context.exposure(),))
assert known.entity_id in exposed.entity_ids
changed_data = known.model_dump()
changed_data["properties"] = {"name": "Ada", "description": "Changed visible value"}
changed_record = EntityRecord.model_validate(changed_data)
changed_context = ContextResult.from_records(
    request=request,
    provider=context.provider,
    entities=(changed_record,),
)
assert changed_context.fingerprint != context.fingerprint
try:
    ExposureSet(scope=effective, contexts=(context.exposure(),) * 3)
except ReadLimitError as error:
    assert error.usage.results == 3
    print("Three tool results exceed the shared two-record allowance.")
else:
    raise AssertionError("each tool must not receive a fresh allowance")

## Proposals and assertion supports are not approved canonical publication

No model runs here. We construct a proposal to demonstrate the interface and verify its
original bytes. Full class/predicate IDs survive unchanged. Run IDs record lineage but
do not define extraction intent. Schema/quote/access validation, resolution and trusted
review are still application responsibilities; constructing a value proves none of them.

In [ ]:
from synirmos.core.extraction import (
    AssertionSupport,
    EntityCandidate,
    EvidenceSpan,
    ExtractionCandidates,
    ExtractionOperation,
    ExtractionProfile,
    ExtractionSubject,
)

subject = ExtractionSubject(
    revision=revisions[0],
    span=span,
    text=line.decode(),
    visibility=visible,
)
subject.verify(raw)
claim = EvidenceSpan(span=span, quote=line.decode())
claim.verify(raw)
profile = ExtractionProfile(
    task=example_artifact("task"),
    model=example_artifact("no-model"),
    prompt=example_artifact("prompt"),
    tools=example_artifact("tools"),
    schema_fingerprint=json_fingerprint({"classes": ["https://example/Person"]}),
    ontology_release=pin.ontology_release,
)
operation = ExtractionOperation(subject=subject, profile=profile, exposure=exposed)
candidate = EntityCandidate(
    operation_id=operation.operation_id,
    local_key="candidate:1",
    name="Ada",
    class_id="https://example/Person",
    evidence=(claim,),
    reuse_entity_id=known.entity_id,
)
proposal = ExtractionCandidates(operation=operation, run_id="run:1", entities=(candidate,))
retry = ExtractionCandidates.model_validate(proposal.model_dump() | {"run_id": "run:retry"})
assert retry.operation.operation_id == operation.operation_id
assert retry.entities[0].candidate_id == candidate.candidate_id
assert proposal.status == ApprovalStatus.PROPOSED
support = AssertionSupport(
    operation_id=operation.operation_id,
    profile_fingerprint=profile.fingerprint,
    subject_id=documents[0].document_id,
    predicate_id="https://example/mentions",
    object_id=known.entity_id,
    evidence=(claim,),
    timing=timing,
    run_ids=(proposal.run_id,),
)
assert support.status == ApprovalStatus.PROPOSED
retry_data = support.model_dump()
retry_data["run_ids"] = (proposal.run_id, retry.run_id)
support_retry = AssertionSupport.model_validate(retry_data)
assert support_retry.assertion_id == support.assertion_id
print("Retry-stable intent/support identity; no graph write or automatic approval.")

## D0 delivery boundary and next slices

D0 supplies neutral values/ports, identity/time/scope/proposal semantics and conformance
tests. Test doubles exercise read-session cleanup and borrowed-reader invalidation.
Real generic storage/profile composition is D1; text capture/parsing/chunking is D2;
scoped readers/context providers are D3. JERE, schema policy, resolution and publication
follow their own slices. This demonstration is not proof of their implementation.

The full gate runs this notebook and notebook 00 as ordinary code cells in fresh
processes, in order, without Jupyter. Extraction quality, real authorization and
deployment scale must be established by later integrations and measurements.